In this example, we convert meshes to NURBS (IGES format) using Rhino3D. The meshes belong to the VSD dataset which is already available in the BoneHub dataset.

First install the `mesh2nurbs-rhino3d` package

In [3]:
# %pip install git+https://github.com/BoneHub/mesh2nurbs-rhino3d.git --upgrade
%pip install -e "C:\Users\AlaviSH\OneDrive - University of Twente\PhD\PythonProjects\mesh2nurbs-rhino3d" --upgrade --force-reinstall

Obtaining file:///C:/Users/AlaviSH/OneDrive%20-%20University%20of%20Twente/PhD/PythonProjects/mesh2nurbs-rhino3d
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for mesh2nurbs_rhino3d (pyproject.toml): started
  Building editable for mesh2nurbs_rhino3d (pyproject.toml): finished with status 'done'
  Created wheel for mesh2nurbs_rhino3d: filename=mesh2nurbs_rhino3d-0.2-0.editable-py3-none-any.whl size=4007 sha256=63edf092c1ab02e3217b043d3764e83fb33cfe752d3ca5bf6f9adecb986b633a
  Stored in directory: C:\Users\AlaviSH\AppData\Loc

In [1]:
from pathlib import Path
from bonehub_data_schema import BoneHubDatasetIO
vsd = BoneHubDatasetIO(Path(r"Z:\BoneHub\BoneHub_Dataset"), 5) # ID of the VSD is 5 in the BoneHub_Dataset

In [ ]:
from pathlib import Path
import tempfile
import shutil

for sub in vsd.subject_info:
    all_meshes = vsd.get_mesh_paths(sub)
    for bonename, mesh_path in all_meshes.items():
        with tempfile.TemporaryDirectory() as tmp:
            output_folder = str(Path(tmp))
            print(f"Processing `{mesh_path}` ...")
            output_file_path = Path(output_folder) / "results" / "CADModel" / (Path(mesh_path).stem + ".iges")
            copy_path = vsd.dataset_path / "NURBS" / mesh_path.parent.name / (output_file_path.name)
            if copy_path.exists():
                print(f"NURBS surface already exists at `{copy_path}`. Skipping generation.")
                if sub.nurbs is None:
                    sub.nurbs = {}
                sub.nurbs[bonename] = 2 # 2 indicates that the NURBS surface is generated by BoneHub
                continue
            !mesh2nurbs --input {mesh_path} --output {output_folder} --nopreprocessing --subdtype 1 --quadremeshlength 2.0 --filetype iges
            if output_file_path.exists():
                print(f"Successfully created NURBS surface at `{output_file_path}`.")
                copy_path.parent.mkdir(parents=True, exist_ok=True)
                shutil.copyfile(output_file_path, vsd.dataset_path / "NURBS" / mesh_path.parent.name / (output_file_path.name))
                if copy_path.exists():
                    print(f"Successfully copied NURBS surface to `{copy_path}`.")
                    if sub.nurbs is None:
                        sub.nurbs = {}
                    sub.nurbs[bonename] = 2 # 2 indicates that the NURBS surface is generated by BoneHub
                else:
                    print(f"WARNING: Failed to copy NURBS surface to `{copy_path}`.")
            else:
                print(f"WARNING: Failed to create NURBS surface for `{mesh_path}`.")

vsd.save_subject_info()
print("successfully updated the subject info with the new NURBS surface information.")

Processing `Z:\BoneHub\BoneHub_Dataset\Dataset_005\Mesh\005_000001\005_000001_SACRUM.stl` ...
NURBS surface already exists at `Z:\BoneHub\BoneHub_Dataset\Dataset_005\NURBS\005_000001\005_000001_SACRUM.iges`. Skipping generation.
Processing `Z:\BoneHub\BoneHub_Dataset\Dataset_005\Mesh\005_000001\005_000001_HIP_LEFT.stl` ...
NURBS surface already exists at `Z:\BoneHub\BoneHub_Dataset\Dataset_005\NURBS\005_000001\005_000001_HIP_LEFT.iges`. Skipping generation.
Processing `Z:\BoneHub\BoneHub_Dataset\Dataset_005\Mesh\005_000001\005_000001_HIP_RIGHT.stl` ...
NURBS surface already exists at `Z:\BoneHub\BoneHub_Dataset\Dataset_005\NURBS\005_000001\005_000001_HIP_RIGHT.iges`. Skipping generation.
Processing `Z:\BoneHub\BoneHub_Dataset\Dataset_005\Mesh\005_000001\005_000001_FEMUR_LEFT.stl` ...
NURBS surface already exists at `Z:\BoneHub\BoneHub_Dataset\Dataset_005\NURBS\005_000001\005_000001_FEMUR_LEFT.iges`. Skipping generation.
Processing `Z:\BoneHub\BoneHub_Dataset\Dataset_005\Mesh\005_000001

In [2]:
if vsd.check_dataset_integrity():
    print("Dataset integrity check passed.")
else:
    print("WARNING: Dataset integrity check failed. Please investigate the issues.")

Dataset integrity check passed.
